# DATA GENERATION

In [6]:
import os
from google.cloud import bigquery
from faker import Faker
import pandas as pd
import random
from datetime import timedelta

fake = Faker()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

assert PROJECT_ID is not None, "Falta PROJECT_ID"
assert DATASET_ID is not None, "Falta DATASET_ID"

client = bigquery.Client(project=PROJECT_ID, location="EU")

fake = Faker()

BASE_ID = 10000


AssertionError: Falta PROJECT_ID

## Clientes

In [2]:
customers = []

for i in range(500):
    customers.append({
        "customer_id": BASE_ID + i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "country": fake.country_code(),
        "city": fake.city(),
        "adquisition_channel": random.choice(["organic","paid_ads","social","referral"]),
        "registration_date": fake.date_time_between(start_date='-2y', end_date='now')
    })

df_customers = pd.DataFrame(customers)


## Categories

In [3]:
categories = [
    {"category_id": 1, "name": "Smartphones", "description": "Mobile phones and smartphone accessories"},
    {"category_id": 2, "name": "Laptops", "description": "Portable computers and ultrabooks"},
    {"category_id": 3, "name": "Tablets", "description": "Tablets and portable touch devices"},
    {"category_id": 4, "name": "Gaming", "description": "Gaming consoles and gaming accessories"},
    {"category_id": 5, "name": "Audio", "description": "Headphones, speakers and audio systems"},
    {"category_id": 6, "name": "Monitors", "description": "Computer monitors and displays"},
    {"category_id": 7, "name": "Smart Home", "description": "Smart home and IoT devices"},
    {"category_id": 8, "name": "Wearables", "description": "Smart watches and wearable technology"},
    {"category_id": 9, "name": "Accessories", "description": "Electronic accessories and peripherals"},
    {"category_id": 10, "name": "Storage", "description": "External drives, SSDs and USB storage"}
]

df_categories = pd.DataFrame(categories)

df_categories

,category_id,name,description
0,1,Smartphones,Mobile phones and smartphone accessories
1,2,Laptops,Portable computers and ultrabooks
2,3,Tablets,Tablets and portable touch devices
3,4,Gaming,Gaming consoles and gaming accessories
4,5,Audio,"Headphones, speakers and audio systems"
5,6,Monitors,Computer monitors and displays
6,7,Smart Home,Smart home and IoT devices
7,8,Wearables,Smart watches and wearable technology
8,9,Accessories,Electronic accessories and peripherals
9,10,Storage,"External drives, SSDs and USB storage"


## Productos

In [7]:
product_rows = []
product_id = 1

electronics_categories = df_categories["name"].tolist()

for category_id, category in enumerate(electronics_categories, start=1):

    for _ in range(7):

        sale_price = round(random.uniform(50, 2500), 2)
        cost_price = round(sale_price * random.uniform(0.55, 0.80), 2)

        product_rows.append({
            "product_id": product_id,
            "category_id": category_id,
            "product_name": f"{fake.company()} {fake.word().title()}",
            "brand": fake.company(),
            "sale_price": sale_price,
            "cost_price": cost_price,
            "stock": random.randint(5, 200),
            "is_active": True
        })

        product_id += 1

df_products = pd.DataFrame(product_rows)

df_products.head()

,product_id,category_id,product_name,brand,sale_price,cost_price,stock,is_active
0,1,1,Nguyen Ltd Whose,Suarez-Higgins,1257.33,834.96,137,True
1,2,1,"Flores, Townsend and Ortiz Present","Singh, Wright and Reyes",1577.63,940.96,170,True
2,3,1,Swanson-Smith Remain,King Group,1145.67,776.59,67,True
3,4,1,Reynolds LLC His,"Medina, Mendez and Welch",1878.82,1322.01,33,True
4,5,1,Fisher-Jones Late,"Allen, Mullins and Hayden",1755.64,1173.01,47,True


## Orders + Items

In [5]:
orders = []
order_items = []

order_id = BASE_ID
order_item_id = BASE_ID

for i in range(2000):
    order_date = fake.date_time_between(start_date='-1y', end_date='now')

    orders.append({
        "order_id": order_id,
        "customer_id": random.choice(df_customers["customer_id"]),
        "order_date": order_date,
        "order_status": "delivered"
    })

    for _ in range(random.randint(2,3)):
        product = random.choice(products)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order_id,
            "product_id": product["product_id"],
            "quantity": random.randint(1,4),
            "unit_price": product["sale_price"]
        })

        order_item_id += 1

    order_id += 1

df_orders = pd.DataFrame(orders)
df_order_items = pd.DataFrame(order_items)


## Pagos

In [6]:
payments = []

for o in orders:
    total = sum([
        oi["quantity"] * oi["unit_price"]
        for oi in order_items if oi["order_id"] == o["order_id"]
    ])

    payments.append({
        "payment_id": o["order_id"],
        "order_id": o["order_id"],
        "payment_method": random.choice(["credit_card","paypal","bank_transfer"]),
        "payment_status": "completed",
        "payment_date": o["order_date"],
        "amount": round(total,2)
    })

df_payments = pd.DataFrame(payments)


## Shipments

In [7]:
shipments = []

for o in orders:
    shipments.append({
        "shipments_id": o["order_id"],
        "order_id": o["order_id"],
        "shipping_status": "delivered",
        "shipping_country": fake.country_code(),
        "shipping_city": fake.city(),
        "delivery_date": o["order_date"] + timedelta(days=random.randint(1,5)),
        "street_address": fake.street_address(),
        "cp_code": fake.postcode(),
        "phone": fake.phone_number()
    })

df_shipments = pd.DataFrame(shipments)


## Reviews

In [8]:
reviews = []
review_id = BASE_ID

for oi in order_items:
    if random.random() < 0.35:
        reviews.append({
            "review_id": review_id,
            "customer_id": random.choice(df_customers["customer_id"]),
            "product_id": oi["product_id"],
            "rating": random.randint(1,5),
            "review_date": fake.date_time_between(start_date='-6m', end_date='now'),
            "comment": fake.sentence()
        })
        review_id += 1

df_reviews = pd.DataFrame(reviews)


## Upload

In [9]:
from google.cloud import bigquery

def upload(df, table_name):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_TRUNCATE"
        )
    )

    job.result()
    print(f"{table_name} añadida correctamente")

upload(df_categories, "categories")
upload(df_customers, "customers")
upload(df_products, "products")
upload(df_orders, "orders")
upload(df_order_items, "order_items")
upload(df_payments, "payments")
upload(df_shipments, "shipments")
upload(df_reviews, "reviews")


categories añadida correctamente
customers añadida correctamente
products añadida correctamente
orders añadida correctamente
order_items añadida correctamente
payments añadida correctamente
shipments añadida correctamente
reviews añadida correctamente
